# 🐲 Automated Stereogram 3D Depth Map Generator (Qwen-Image-2.1 + Apple Depth Pro 2)

A streamlined **Automated Batch Loop** notebook for Google Colab (NVIDIA A100) that generates paired 3D figures and stereogram-ready grayscale depth maps directly in Python (no ComfyUI or tunnels required):

### 🔄 Pipeline Overview:
1. **Text LLM Prompt Generator (Optional Ollama `qwen2.5:7b` / Procedural Fallback)**: Crafts diverse 3D stereogram figure prompts within your chosen `THEME` (forward-protruding features, isolated on pure black background).
2. **Qwen-Image-2.1 (`diffusers`)**: Renders high-resolution 3D figures with clean silhouettes.
3. **Apple Depth Pro 2 (`apple-aiml-research/ml-depth-pro-2`)**: Estimates sharp, full-resolution metric depth (with automatic focal length estimation via `apple/DepthPro2-focal`) and converts metric distance into stereoscopic inverse depth ($255 = \text{near}, 0 = \text{far}$).
4. **Google Drive Persistence**: Automatically saves paired 3D figures and depth maps to `My Drive/Stereogram_Depthmaps`.

> **License Note:** Depth Pro 2 code and model weights (`apple/DepthPro2-*`) are released under the **Apple Machine Learning Research Model License** (non-commercial research and academic use).

## 1. Verify GPU & Python Environment
Ensure you have an NVIDIA A100 GPU connected (`Runtime` -> `Change runtime type` -> select `A100 GPU`). Depth Pro 2 also requires Python $\ge 3.11$ (Colab defaults to Python 3.12).

In [ ]:
!nvidia-smi
import sys
print("Python:", sys.version)

## 2. Mount Google Drive for Outputs
All generated 3D figures and stereogram depth maps will be saved directly to your Google Drive (`MyDrive/Stereogram_Depthmaps`).

In [ ]:
import os
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Stereogram_Depthmaps"
os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "3D_Figures"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_OUTPUT_DIR, "Stereogram_Depthmaps"), exist_ok=True)
print(f"✓ Google Drive mounted! Output directories ready at:\n  {DRIVE_OUTPUT_DIR}")

## 3. Install Dependencies (`Qwen-Image-2.1` + `Apple Depth Pro 2`)
Installs PyTorch ($\ge 2.7$), Apple's `ml-depth-pro-2`, modern `diffusers`/`transformers` (with `huggingface_hub` pinned `< 2.0` and `pillow` pinned `< 13.0` to avoid Colab resolver conflicts), and `zstd` for optional Ollama setup.

In [ ]:
!pip install -q -U "torch>=2.7" torchvision torchaudio
!pip install -q "git+https://github.com/apple-aiml-research/ml-depth-pro-2" "huggingface_hub>=1.31.0,<2.0" "pillow>=11.3,<13.0"
!pip install -q -U "transformers>=5.17.0" accelerate safetensors torchao scipy einops kornia "huggingface_hub>=1.31.0,<2.0" "pillow>=11.3,<13.0" git+https://github.com/huggingface/diffusers.git
!apt-get -y update -qq && apt-get -y install -qq zstd

import torch
import diffusers
import transformers
import depth_pro_2

print(f"✓ PyTorch:       {torch.__version__} (CUDA available: {torch.cuda.is_available()})")
print(f"✓ Diffusers:     {diffusers.__version__}")
print(f"✓ Transformers:  {transformers.__version__}")
print("✓ Depth Pro 2:   imported successfully!")

## 4. (Optional) Run Free Local Text LLM (Ollama) in Colab
Runs a local `qwen2.5:7b` model on port `11434` to automatically craft creative 3D stereogram figure prompts for your `THEME`. If you skip this cell (or set `USE_OLLAMA_PROMPTS = False`), the batch loop automatically uses the built-in procedural prompt generator.

In [ ]:
import subprocess
import time
import urllib.request
import os

print("Installing Ollama daemon...")
!curl -fsSL https://ollama.com/install.sh | sh

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.run("pkill -9 ollama 2>/dev/null", shell=True)
subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
time.sleep(2)

for _ in range(15):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=1) as resp:
            if resp.status == 200:
                print("✓ Ollama is running and healthy on port 11434!")
                break
    except Exception:
        time.sleep(1)

print("Pulling Qwen2.5 7B text model for stereogram prompt generation...")
!ollama pull qwen2.5:7b
print("✓ Local prompt LLM ready!")

## 5. Automated Batch Loop (`Qwen-Image-2.1` → `Apple Depth Pro 2`)
Use the form controls below to adjust prompts, lighting, background isolation (`BiRefNet`), relief contrast, and Depth Pro 2 model size. The Python implementation stays hidden behind the form view (`⋮ → Show code` to inspect or edit it).

In [ ]:
#@title 🎛️ Batch Configuration & Run { display-mode: "form" }
#@markdown ### 📝 Prompt & Lighting Settings
THEME = "Animals from Africa" #@param {type:"string"}
CUSTOM_PROMPT = "" #@param {type:"string"}
RENDER_STYLE = "natural" #@param ["natural", "sculpture"]
LIGHTING_MODE = "bright_softbox" #@param ["bright_softbox", "studio_rim"]
STUDIO_BACKDROP = "pure_black" #@param ["pure_black", "neutral_gray", "pure_white"]
USE_OLLAMA_PROMPTS = True #@param {type:"boolean"}
#@markdown ### 📐 Output & Depth Pro 2 Settings
ASPECT_RATIO = "3:2" #@param ["3:2", "16:9", "4:3", "1:1", "2:3", "9:16"]
NUM_IMAGES = 5 #@param {type:"integer"}
RESOLUTION_SCALE = 0.5 #@param {type:"slider", min:0.25, max:1.0, step:0.25}
DEPTHPRO2_MODEL = "apple/DepthPro2-H" #@param ["apple/DepthPro2-H", "apple/DepthPro2-L", "apple/DepthPro2-B", "apple/DepthPro2-S"]
BACKGROUND_ISOLATION = "birefnet" #@param ["birefnet", "none", "border_black"]
RELIEF_ENHANCE = 0.35 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
DEPTH_GAMMA = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.1}
#@markdown ### ⚙️ Advanced Sampling & Model Cache
NUM_INFERENCE_STEPS = 30 #@param {type:"integer"}
TRUE_CFG_SCALE = 4.0 #@param {type:"slider", min:1.0, max:8.0, step:0.5}
RELOAD_MODELS = False #@param {type:"boolean"}

import os
import sys
import time
import json
import random
import subprocess
import importlib.util
import urllib.request
from collections import deque
from functools import partial

import numpy as np
import torch
from PIL import Image, ImageFilter
from IPython.display import display

DRIVE_DIR = globals().get('DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/Stereogram_Depthmaps')
FIG_DIR = os.path.join(DRIVE_DIR, '3D_Figures')
DEPTH_DIR = os.path.join(DRIVE_DIR, 'Stereogram_Depthmaps')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DEPTH_DIR, exist_ok=True)

# ── Dimension calculation for arbitrary aspect ratios (multiples of 32) ──
ASPECT_RATIOS = {
    "3:2":  (2528, 1696),
    "16:9": (2752, 1536),
    "4:3":  (2400, 1792),
    "1:1":  (2048, 2048),
    "2:3":  (1696, 2528),
    "9:16": (1536, 2752),
}

def get_dimensions(aspect="3:2", scale=0.5):
    base_w, base_h = ASPECT_RATIOS.get(aspect, (2528, 1696))
    w = max(256, int(round((base_w * scale) / 32)) * 32)
    h = max(256, int(round((base_h * scale) / 32)) * 32)
    return w, h

width, height = get_dimensions(ASPECT_RATIO, RESOLUTION_SCALE)

LIGHTING_PHRASES = {
    "bright_softbox": "bright even 360-degree studio softbox lighting, soft bounce fill light illuminating all undersides and folds with zero dark shadows",
    "studio_rim": "dramatic studio rim lighting",
}

BACKDROP_PHRASES = {
    "pure_black": "isolated on solid pure black background",
    "neutral_gray": "isolated on a clean uniform seamless medium-gray studio backdrop, floating centered subject, zero floor shadows",
    "pure_white": "isolated on a clean uniform seamless pure white studio backdrop, floating centered subject, zero floor shadows",
}

lighting_phrase = LIGHTING_PHRASES.get(LIGHTING_MODE, LIGHTING_PHRASES["bright_softbox"])
backdrop_phrase = BACKDROP_PHRASES.get(STUDIO_BACKDROP, BACKDROP_PHRASES["pure_black"])

# ── Smart Theme-Based Prompt Generator ──
MATERIALS = [
    "matte gray sculptor clay",
    "chiseled white Carrara marble",
    "light travertine limestone",
    "fine-grain volcanic basalt stone",
    "matte ceramic porcelain",
    "warm alabaster stone"
]

FALLBACK_SUBJECTS = {
    "animals from africa": [
        "a charging African bull elephant with massive curved ivory tusks and raised trunk pointing forward toward the viewer",
        "a roaring male lion in mid-leap with open jaws and outstretched claws facing the camera",
        "a white rhinoceros with a huge sharp nasal horn thrust forward in aggressive stance",
        "a coiled cobra with wide open mouth and spread hood poised forward",
        "a Nile crocodile with gaping jaws showing sharp teeth angled toward the viewer",
        "a spotted leopard crouched on a forward rock with front paws extending forward",
        "a silverback gorilla pounding its chest with muscular arms reaching forward"
    ],
    "mythological": [
        "an ancient European dragon with spread wings and sharp horned snout protruding toward the viewer",
        "a majestic winged Pegasus rearing up with front hooves striking forward",
        "a Greek Chimera with lion head, goat horns, and serpent tail arching forward",
        "a ferocious Cerberus with three snarling heads facing the camera",
        "a sea leviathan rising with jagged jaws and curling tentacles reaching forward"
    ],
    "human": [
        "an athletic sprinter in mid-stride leaning toward the viewer with one hand reaching forward",
        "a classical ballet dancer in dynamic leap with one arm extended toward the camera",
        "a violinist passionately playing with violin scroll and bow angled forward toward the viewer",
        "a rock climber reaching upward and outward with outstretched hand and knee",
        "a martial artist mid-kick with forward-extended palm strike and balanced stance",
        "a portrait of an expressive elder artisan holding up a hand tool toward the camera",
        "an orchestral conductor leaning forward with raised hands and baton reaching toward the viewer"
    ]
}

HUMAN_THEME_KEYWORDS = ("human", "people", "person", "portrait", "dancer", "athlete", "musician", "man", "woman")

def fallback_prompt(theme: str, index: int, render_style: str = "natural") -> str:
    theme_lower = theme.lower()
    subject = None
    if any(k in theme_lower for k in HUMAN_THEME_KEYWORDS):
        subject = FALLBACK_SUBJECTS["human"][index % len(FALLBACK_SUBJECTS["human"])]
    else:
        for key, subjects in FALLBACK_SUBJECTS.items():
            if key in theme_lower:
                subject = subjects[index % len(subjects)]
                break
    if not subject:
        subject = f"a realistic {theme} subject (#{index + 1}) in a dynamic three-quarter forward pose with natural foreground elements reaching toward the camera"

    if render_style == "natural":
        return f"A full 3D photorealistic digital render of {subject}, dynamic volumetric 3D posture with natural forward depth planes, authentic realistic textures and clearly visible surface details, {lighting_phrase}, {backdrop_phrase}, octane render, clean silhouette, 8k"
    mat = MATERIALS[index % len(MATERIALS)]
    return f"A highly detailed 3D digital sculpture of {subject}, volumetric relief with natural forward depth planes, {mat} finish, {lighting_phrase}, {backdrop_phrase}, octane render, clean silhouette, 8k"

def generate_batch_prompts(theme: str, count: int, render_style: str = "natural") -> list:
    """Uses Ollama LLM to brainstorm `count` diverse, distinct 3D stereogram figure prompts within `theme`."""
    if render_style == "natural":
        style_rules = (
            f"3. NATURAL REALISTIC RENDER (NO CLAY/STONE): Render each subject with authentic materials and textures strictly appropriate for \"{theme}\" "
            "(such as natural human skin, hair, and clothing for people; or realistic metal, wood, glass, fabric, or wildlife textures only when the theme calls for them). "
            "Use purely positive visual descriptions matching the theme."
        )
        sample = (
            f"A full 3D photorealistic render of [specific subject strictly matching {theme}] with dynamic volumetric 3D posture, "
            f"natural foreground elements reaching toward the camera, authentic realistic textures and fine details, "
            f"{lighting_phrase}, {backdrop_phrase}, octane render, clean silhouette, 8k"
        )
    else:
        style_rules = (
            "3. SCULPTURE AESTHETIC: Specify physical sculpture materials (matte clay, marble, limestone, porcelain) while keeping anatomy strictly faithful to the theme."
        )
        sample = (
            f"A 3D digital sculpture of [specific subject strictly matching {theme}] with dynamic volumetric 3D posture, "
            f"natural foreground elements reaching toward the camera, matte gray clay finish, "
            f"{lighting_phrase}, {backdrop_phrase}, octane render, clean silhouette, 8k"
        )

    if USE_OLLAMA_PROMPTS:
        instruction = f"""You are an expert 3D stereogram prompt designer.
Theme: "{theme}".
Format: {ASPECT_RATIO}.
Style Mode: {render_style}.
Generate a list of exactly {count} completely different, unique 3D figures that fit this theme.

CRITICAL RULES:
1. SPECIFIC DISTINCT SUBJECTS: Each of the {count} figures must be a completely different subject belonging strictly to "{theme}". All {count} items MUST belong directly to "{theme}". (Do NOT repeat the same subject, and do NOT include unrelated subjects).
2. STEREOGRAM 3D DEPTH: Describe natural, theme-authentic forward-reaching 3D depth planes facing the camera (e.g. outstretched arms or hands, leaning posture, held props or tools, or layered three-quarter perspective) that strictly fit "{theme}". Keep anatomy, proportions, and features completely faithful to "{theme}".
{style_rules}
4. LIGHTING & BACKDROP: Must specify {lighting_phrase} and {backdrop_phrase} with clean edges and zero background clutter.

Return ONLY a valid JSON list of {count} prompt strings.
Format:
["{sample}", ...]"""

        try:
            req = urllib.request.Request(
                "http://127.0.0.1:11434/api/generate",
                data=json.dumps({
                    "model": "qwen2.5:7b",
                    "prompt": instruction,
                    "format": "json",
                    "stream": False,
                    "options": {"temperature": 0.85, "top_p": 0.9}
                }).encode('utf-8'),
                headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=90) as r:
                raw = json.loads(r.read().decode('utf-8')).get('response', '').strip()
                parsed = []
                try:
                    data = json.loads(raw)
                    if isinstance(data, list):
                        parsed = [str(x).strip() for x in data if len(str(x).strip()) > 20]
                    elif isinstance(data, dict):
                        for k in ["prompts", "figures", "items", "images", "list"]:
                            if k in data and isinstance(data[k], list):
                                parsed = [str(x).strip() for x in data[k] if len(str(x).strip()) > 20]
                                break
                        if not parsed:
                            parsed = [str(v).strip() for v in data.values() if isinstance(v, str) and len(v.strip()) > 20]
                except Exception:
                    for line in raw.splitlines():
                        cleaned = line.strip().lstrip('0123456789.-*• "').rstrip('",')
                        if len(cleaned) > 25:
                            parsed.append(cleaned)

                if parsed:
                    cleaned_prompts = []
                    for p in parsed:
                        p = p.replace("**", "").replace("Prompt:", "").strip().strip('"')
                        if "[name of" in p or "[insert" in p:
                            continue
                        if "elephant" in p.lower() and "elephant" not in theme.lower() and "africa" not in theme.lower():
                            continue
                        if len(p) > 25:
                            cleaned_prompts.append(p)
                    while len(cleaned_prompts) < count:
                        cleaned_prompts.append(fallback_prompt(theme, len(cleaned_prompts), render_style))
                    return cleaned_prompts[:count]
        except Exception as e:
            print(f"  [Notice] Ollama prompt generation failed ({e}); using procedural theme generator.")

    return [fallback_prompt(theme, i, render_style) for i in range(count)]

def _border_black_mask(rgb_image: Image.Image, threshold: int = 3) -> np.ndarray:
    rgb_arr = np.asarray(rgb_image.convert("RGB"))
    dark = np.max(rgb_arr, axis=2) <= threshold
    try:
        from scipy import ndimage
        labeled, num_features = ndimage.label(dark)
        if num_features == 0:
            return np.zeros_like(dark, dtype=bool)
        border_labels = np.unique(np.concatenate([
            labeled[0, :], labeled[-1, :], labeled[:, 0], labeled[:, -1]
        ]))
        border_labels = border_labels[border_labels > 0]
        return np.isin(labeled, border_labels) if len(border_labels) > 0 else np.zeros_like(dark, dtype=bool)
    except ImportError:
        h, w = dark.shape
        mask = np.zeros((h, w), dtype=bool)
        q = deque()
        for x in range(w):
            if dark[0, x] and not mask[0, x]: mask[0, x] = True; q.append((0, x))
            if dark[h - 1, x] and not mask[h - 1, x]: mask[h - 1, x] = True; q.append((h - 1, x))
        for y in range(h):
            if dark[y, 0] and not mask[y, 0]: mask[y, 0] = True; q.append((y, 0))
            if dark[y, w - 1] and not mask[y, w - 1]: mask[y, w - 1] = True; q.append((y, w - 1))
        while q:
            cy, cx = q.popleft()
            for ny, nx in ((cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)):
                if 0 <= ny < h and 0 <= nx < w and dark[ny, nx] and not mask[ny, nx]:
                    mask[ny, nx] = True
                    q.append((ny, nx))
        return mask

def compute_birefnet_mask(birefnet_model, rgb_image: Image.Image, device: str) -> np.ndarray:
    from torchvision import transforms
    tf = transforms.Compose([
        transforms.Resize((1024, 1024)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    orig_w, orig_h = rgb_image.size
    param_dtype = next(birefnet_model.parameters()).dtype
    inp = tf(rgb_image.convert("RGB")).unsqueeze(0).to(device=device, dtype=param_dtype)
    with torch.inference_mode():
        preds = birefnet_model(inp)[-1].sigmoid().float().cpu()
    pred = np.clip(preds[0].squeeze().numpy() * 255.0, 0.0, 255.0).astype(np.uint8)
    mask_pil = Image.fromarray(pred, mode="L").resize((orig_w, orig_h), Image.BILINEAR)
    return np.asarray(mask_pil, dtype=np.float32) / 255.0

# ── Depth Pro 2 Metric Depth -> Stereogram Inverse-Depth Converter ──
def metric_depth_to_stereogram_map(
    depth_tensor: torch.Tensor,
    rgb_image: Image.Image,
    isolation_mode: str = "birefnet",
    fg_alpha_override: np.ndarray = None,
    relief_enhance: float = 0.35,
    depth_gamma: float = 1.0,
    black_threshold: int = 3,
    percentile_clip: float = 99.5,
) -> tuple:
    """
    Converts Depth Pro 2 metric depth (meters, smaller = closer) into an 8-bit
    stereogram depth map (255 = nearest foreground, 0 = farthest background).
    """
    depth_m = depth_tensor.detach().float().cpu().numpy()
    depth_m = np.squeeze(depth_m)

    valid = np.isfinite(depth_m) & (depth_m > 1e-3)
    if not np.any(valid):
        return Image.fromarray(np.zeros(depth_m.shape, dtype=np.uint8), mode="L"), 0.0, 0.0

    max_valid = float(np.max(depth_m[valid]))
    depth_m = np.where(valid, depth_m, max_valid)
    depth_m = np.clip(depth_m, 1e-3, 1e4)

    near_m = float(np.percentile(depth_m, 0.2))
    far_m = float(np.percentile(depth_m, 99.8))

    # Stereoscopic disparity scales linearly with inverse depth (1 / distance)
    inv_depth = 1.0 / depth_m

    fg_alpha = None
    if isolation_mode == "birefnet" and fg_alpha_override is not None:
        fg_alpha = np.clip(fg_alpha_override.astype(np.float32), 0.0, 1.0)
    elif isolation_mode == "border_black":
        bg_mask = _border_black_mask(rgb_image, threshold=black_threshold)
        if np.any(~bg_mask) and np.mean(bg_mask) < 0.98:
            fg_u8 = ((~bg_mask).astype(np.uint8) * 255)
            fg_alpha = np.asarray(Image.fromarray(fg_u8, mode="L").filter(ImageFilter.GaussianBlur(radius=0.8)), dtype=np.float32) / 255.0
            fg_alpha = np.where(bg_mask, np.minimum(fg_alpha, 0.25), 1.0)

    has_fg = fg_alpha is not None and np.any(fg_alpha > 0.5) and np.mean(fg_alpha > 0.5) > 0.01
    if has_fg:
        fg_vals = inv_depth[fg_alpha > 0.5]
        fg_low = float(np.percentile(fg_vals, 0.5))
        fg_high = float(np.percentile(fg_vals, percentile_clip))
        if fg_high <= fg_low:
            fg_low = float(fg_vals.min())
            fg_high = float(fg_vals.max())

        linear = np.clip((inv_depth - fg_low) / max(fg_high - fg_low, 1e-6), 0.0, 1.0)
        if relief_enhance > 0.0 and fg_vals.size > 256:
            qs = np.linspace(0.0, 1.0, 256)
            q_vals = np.maximum.accumulate(np.quantile(fg_vals, qs))
            cdf_map = np.interp(inv_depth, q_vals, qs)
            fg_rel = (1.0 - relief_enhance) * linear + relief_enhance * cdf_map
        else:
            fg_rel = linear

        if abs(depth_gamma - 1.0) > 1e-3:
            fg_rel = np.power(np.clip(fg_rel, 0.0, 1.0), depth_gamma)

        # Keep clean z=0 backdrop with a 12% base elevation for the farthest foreground rim
        base_offset = 0.12
        norm = (base_offset + (1.0 - base_offset) * np.clip(fg_rel, 0.0, 1.0)) * fg_alpha
    else:
        d_min = float(np.percentile(inv_depth, 0.2))
        d_max = float(np.percentile(inv_depth, percentile_clip))
        if d_max <= d_min:
            d_min = float(inv_depth.min())
            d_max = float(inv_depth.max())
        linear = np.clip((inv_depth - d_min) / max(d_max - d_min, 1e-6), 0.0, 1.0)
        if relief_enhance > 0.0:
            qs = np.linspace(0.0, 1.0, 256)
            q_vals = np.maximum.accumulate(np.quantile(inv_depth, qs))
            cdf_map = np.interp(inv_depth, q_vals, qs)
            norm = (1.0 - relief_enhance) * linear + relief_enhance * cdf_map
        else:
            norm = linear
        if abs(depth_gamma - 1.0) > 1e-3:
            norm = np.power(np.clip(norm, 0.0, 1.0), depth_gamma)

    depth_uint8 = np.round(np.clip(norm, 0.0, 1.0) * 255.0).astype(np.uint8)
    return Image.fromarray(depth_uint8, mode="L"), near_m, far_m

def flatten_rgba_on_black(img: Image.Image) -> Image.Image:
    if img.mode == "RGBA":
        bg = Image.new("RGB", img.size, (0, 0, 0))
        bg.paste(img, mask=img.split()[3])
        return bg
    return img.convert("RGB")

# ── Step 1: Plan Batch Prompts ──
if CUSTOM_PROMPT.strip():
    prompts = [CUSTOM_PROMPT.strip() for _ in range(max(1, int(NUM_IMAGES)))]
    print(f"📝 Using CUSTOM_PROMPT ({len(prompts)} variation(s)):")
    print(f"   \"{prompts[0]}\"\n")
else:
    print(f"🧠 Brainstorming {NUM_IMAGES} diverse 3D prompts for theme: \"{THEME}\" ({RENDER_STYLE}, {LIGHTING_MODE})...")
    prompts = generate_batch_prompts(THEME, int(NUM_IMAGES), RENDER_STYLE)
    print(f"✓ Planned {len(prompts)} unique figure prompts:\n")
    for idx, p in enumerate(prompts, 1):
        print(f"   [{idx}/{len(prompts)}] {p[:120]}...")
    print()

# ── Step 2: Load & Cache Qwen-Image-2.1, Apple Depth Pro 2, and BiRefNet ──
try:
    import torchao.quantization
    if not hasattr(torchao.quantization, "FqnToConfig"):
        class FqnToConfig:
            pass
        torchao.quantization.FqnToConfig = FqnToConfig
except Exception:
    pass

from diffusers import QwenImage21Pipeline
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from depth_pro_2 import DepthPro2, load_rgb
from depth_pro_2.model import to_normalized_tensor
from depth_pro_2.network.fov import FOVEstimator

device_str = "cuda" if torch.cuda.is_available() else "cpu"
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0
model_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32)
keep_aux_on_gpu = torch.cuda.is_available() and total_vram_gb > 24

cache = globals().setdefault("_STEREOCRAFT_CACHE", {})
if RELOAD_MODELS:
    cache.clear()

print("=" * 70)
print("🚀 STEREOGRAM DEPTH MAP BATCH GENERATOR (Depth Pro 2 Edition)")
print(f"  Mode:            {'Custom Prompt' if CUSTOM_PROMPT.strip() else THEME}")
print(f"  Lighting / BG:   {LIGHTING_MODE} | {STUDIO_BACKDROP} | mask={BACKGROUND_ISOLATION}")
print(f"  Aspect Ratio:    {ASPECT_RATIO} ({width}x{height})")
print(f"  Depth Model:     {DEPTHPRO2_MODEL} ({model_dtype})")
print(f"  Relief / Gamma:  enhance={RELIEF_ENHANCE:.2f}, gamma={DEPTH_GAMMA:.1f}")
print(f"  GPU / VRAM:      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} ({total_vram_gb:.1f} GB)")
print("=" * 70)

if "qwen_pipe" not in cache:
    print("\n📦 [1/3] Loading Qwen-Image-2.1 Pipeline...")
    qwen_pipe = QwenImage21Pipeline.from_pretrained("Qwen/Qwen-Image-2.1", torch_dtype=model_dtype)
    if total_vram_gb > 50:
        qwen_pipe.to("cuda")
        print("  ✓ Qwen-Image-2.1 loaded directly into VRAM")
    elif torch.cuda.is_available():
        qwen_pipe.enable_model_cpu_offload()
        print("  ✓ Qwen-Image-2.1 loaded with CPU offloading")
    cache["qwen_pipe"] = qwen_pipe
else:
    qwen_pipe = cache["qwen_pipe"]
    print("\n✓ [1/3] Reusing cached Qwen-Image-2.1 Pipeline")

if cache.get("depth_model_id") != DEPTHPRO2_MODEL or "depth_model" not in cache:
    print(f"\n📦 [2/3] Loading Apple Depth Pro 2 ({DEPTHPRO2_MODEL} + apple/DepthPro2-focal)...")
    depth_model = DepthPro2.from_pretrained(DEPTHPRO2_MODEL, precision=model_dtype)
    focal_weights = hf_hub_download("apple/DepthPro2-focal", "model.safetensors")
    depth_model.fov = FOVEstimator(model_dtype)
    depth_model.fov.load_state_dict(load_file(focal_weights))
    depth_model = depth_model.eval()
    if keep_aux_on_gpu:
        depth_model = depth_model.to("cuda")
    cache["depth_model"] = depth_model
    cache["depth_model_id"] = DEPTHPRO2_MODEL
    print("  ✓ Depth Pro 2 ready")
else:
    depth_model = cache["depth_model"]
    print(f"\n✓ [2/3] Reusing cached {DEPTHPRO2_MODEL}")

birefnet_model = None
if BACKGROUND_ISOLATION == "birefnet":
    if "birefnet_model" not in cache:
        print("\n📦 [3/3] Loading ZhengPeng7/BiRefNet semantic foreground matting...")
        missing_pkgs = [p for p in ("einops", "kornia") if importlib.util.find_spec(p) is None]
        if missing_pkgs:
            print(f"  -> Installing missing matting dependencies: {', '.join(missing_pkgs)}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs], check=False)
        try:
            from transformers import AutoModelForImageSegmentation
            birefnet_model = AutoModelForImageSegmentation.from_pretrained("ZhengPeng7/BiRefNet", trust_remote_code=True).float().eval()
            if keep_aux_on_gpu:
                birefnet_model = birefnet_model.to("cuda")
            cache["birefnet_model"] = birefnet_model
            print("  ✓ BiRefNet ready (prevents dark shadow cutouts!)")
        except Exception as e:
            print(f"  ⚠️ Could not load BiRefNet ({e}); continuing with unmasked Depth Pro 2 output.")
            birefnet_model = None
    else:
        birefnet_model = cache["birefnet_model"].float()
        print("\n✓ [3/3] Reusing cached BiRefNet")

transform = partial(to_normalized_tensor, device=torch.device(device_str))

if LIGHTING_MODE == "bright_softbox":
    negative_prompt = (
        "blurry, low quality, flat, noisy, cluttered background, multiple subjects, "
        "deep shadows, crushed blacks, pitch black shadows, silhouette, underexposed, dark crevices, "
        "text, watermark, cropped, floor shadow, ground plane"
    )
else:
    negative_prompt = (
        "blurry, low quality, flat, noisy, cluttered background, multiple subjects, "
        "text, watermark, cropped, noisy texture in background"
    )

# ── Step 3: Automated Batch Generation Loop ──
for idx, prompt in enumerate(prompts, 1):
    seed = random.randint(1, 2147483647)
    print(f"\n[{idx}/{len(prompts)}] ----------------------------------------------------")
    print(f"  📝 Prompt:\n     \"{prompt}\"")

    # A. Render 3D figure with Qwen-Image-2.1
    print(f"  🎨 Rendering 3D figure (Seed: {seed})...")
    t0 = time.time()
    gen = torch.Generator(device=device_str).manual_seed(seed)
    fig_raw = qwen_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=int(NUM_INFERENCE_STEPS),
        true_cfg_scale=float(TRUE_CFG_SCALE),
        generator=gen,
    ).images[0]
    fig_img = flatten_rgba_on_black(fig_raw)
    render_s = time.time() - t0

    fig_path = os.path.join(FIG_DIR, f"figure_{idx:03d}_seed{seed}.png")
    fig_img.save(fig_path)
    print(f"     ✓ Rendered in {render_s:.2f}s -> {fig_path}")

    # B. Semantic Foreground Matting (BiRefNet) if enabled
    fg_alpha_override = None
    eff_isolation = BACKGROUND_ISOLATION
    if BACKGROUND_ISOLATION == "birefnet":
        if birefnet_model is not None:
            if not keep_aux_on_gpu and torch.cuda.is_available():
                birefnet_model = birefnet_model.to("cuda")
            fg_alpha_override = compute_birefnet_mask(birefnet_model, fig_img, device_str)
            if not keep_aux_on_gpu and torch.cuda.is_available():
                birefnet_model = birefnet_model.to("cpu")
                torch.cuda.empty_cache()
        else:
            eff_isolation = "none"

    # C. Estimate Metric Depth with Apple Depth Pro 2 & Convert to Stereogram Map
    print(f"  📐 Estimating metric depth with {DEPTHPRO2_MODEL}...")
    t1 = time.time()
    if not keep_aux_on_gpu and torch.cuda.is_available():
        torch.cuda.empty_cache()
        depth_model = depth_model.to("cuda")

    loaded = load_rgb(fig_path)
    image_np = loaded[0]
    f_px = loaded[-1] if len(loaded) >= 2 else None

    with torch.inference_mode():
        prediction = depth_model.infer(transform(image_np), f_px=f_px)

    if not keep_aux_on_gpu and torch.cuda.is_available():
        depth_model = depth_model.to("cpu")
        torch.cuda.empty_cache()

    depth_img, near_m, far_m = metric_depth_to_stereogram_map(
        prediction["depth"],
        fig_img,
        isolation_mode=eff_isolation,
        fg_alpha_override=fg_alpha_override,
        relief_enhance=float(RELIEF_ENHANCE),
        depth_gamma=float(DEPTH_GAMMA),
    )
    depth_s = time.time() - t1

    est_focal = prediction.get("focallength_px")
    focal_val = float(est_focal.item() if torch.is_tensor(est_focal) else est_focal) if est_focal is not None else 0.0

    depth_path = os.path.join(DEPTH_DIR, f"depthmap_{idx:03d}_seed{seed}.png")
    depth_img.save(depth_path)
    print(f"     ✓ Depth Pro 2 finished in {depth_s:.2f}s (range: {near_m:.2f}m .. {far_m:.2f}m, f_px: {focal_val:.1f}px)")
    print(f"     ✓ Saved stereogram depth map -> {depth_path}")

    display(depth_img)
    display(fig_img)

print("\n" + "=" * 70)
print(f"🎉 Batch generation finished! {len(prompts)} stereogram depth maps created.")
print(f"   Outputs ready in: {DRIVE_DIR}")
print("=" * 70)